In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd

import osmnx as ox

import libpysal
import momepy as mm

from tqdm.auto import tqdm

import scipy as sp
from scipy.cluster import hierarchy
from sklearn import preprocessing
from sklearn.mixture import GaussianMixture

import mapclassify
import matplotlib.pyplot as plt
import seaborn as sns

# **COMPONENTS**

## **Brooklyn Buildings & Streets**

In [ ]:
boroughs = gpd.read_file('data/boroughs.zip')
boroughs = gpd.GeoDataFrame(boroughs[['boro_name', 'geometry']]).to_crs('EPSG:26918')

In [ ]:
buildings = gpd.read_file('data/buildings.zip')

buildings = buildings[['heightroof', 'geometry']]
buildings = buildings.rename(columns={'heightroof':'height'})
buildings = buildings.dropna()

buildings = gpd.GeoDataFrame(buildings)
buildings = buildings.to_crs('EPSG:26918')
buildings = buildings.sjoin(boroughs, how='left', predicate='within')

buildings_bk = gpd.GeoDataFrame(buildings[buildings.boro_name == 'Brooklyn'][['height', 'geometry']])
buildings_bk = buildings_bk.set_crs('EPSG:26918')

buildings_bk['uID'] = mm.unique_id(buildings_bk)

In [ ]:
streets = ox.graph_from_place('New York, New York', network_type='drive')
streets = ox.utils_graph.graph_to_gdfs(streets, nodes=False, edges=True, node_geometry=False, fill_edge_geometry=True)
streets = gpd.GeoDataFrame(streets[['geometry']])
streets = streets.to_crs('EPSG:26918')

streets = streets.sjoin(boroughs, how='left', predicate='within')

streets_bk = gpd.GeoDataFrame(streets[streets.boro_name == 'Brooklyn'][['geometry']]).reset_index(drop=True)
streets_bk = streets_bk.set_crs('EPSG:26918')

streets_bk = mm.remove_false_nodes(streets_bk)
streets_bk.geometry = mm.close_gaps(streets_bk, .25)
streets_bk = mm.extend_lines(streets_bk, tolerance=.2)

In [ ]:
streets_bk['nID'] = mm.unique_id(streets_bk)
buildings_bk['nID'] = mm.get_network_id(buildings_bk, streets_bk, 'nID', min_size=300, verbose=False)
buildings_bk = buildings_bk.dropna()

buildings_bk.to_file('data/buildings_bk.shp')
streets_bk.to_file('data/streets_bk.shp')

## **Brooklyn Tessellation**

In [ ]:
limit_bk = mm.buffered_limit(buildings_bk)
enclosures_bk = mm.enclosures(streets_bk, limit=gpd.GeoSeries([limit_bk]))

tessellation_bk = mm.Tessellation(buildings_bk, unique_id='uID', enclosures=enclosures_bk, use_dask=True).tessellation
tessellation_bk = tessellation_bk.dropna()

In [ ]:
tessellation_bk = tessellation_bk.merge(buildings_bk[['uID', 'nID']], on='uID', how='left')
tessellation_bk.to_file('data/tessellation_bk.shp')

## **Brooklyn Blocks**

In [ ]:
limit_bk = mm.buffered_limit(buildings_bk)
extended_bk = mm.extend_lines(streets_bk, tolerance=120, target=gpd.GeoSeries([limit_bk.boundary]), barrier=buildings_bk)

In [ ]:
blocks_bk = mm.Blocks(tessellation_bk, edges=extended_bk, buildings=buildings_bk, id_name='bID', unique_id='uID')

blocks_bk_df = blocks_bk.blocks
blocks_bk_df.to_file('data/blocks_bk.shp')

In [ ]:
buildings_bk['bID'] = blocks_bk.buildings_id.values
tessellation_bk['bID'] = blocks_bk.tessellation_id.values

buildings_bk.to_file('data/buildings_bk.shp')
tessellation_bk.to_file('data/tessellation_bk.shp')

# **MORPHOMETRICS**

In [ ]:
buildings_bk = gpd.read_file('data/buildings_bk.shp')
tessellation_bk = gpd.read_file('data/tessellation_bk.shp')
streets_bk = gpd.read_file('data/streets_bk.shp')
blocks_bk = gpd.read_file('data/blocks_bk.shp')

In [ ]:
buildings_bk['buildingHeight'] = buildings_bk['height'] * 0.304800
buildings_bk['buildingArea'] = mm.Area(buildings_bk).series
buildings_bk['buildingFloorArea'] = (buildings_bk['buildingHeight'] / 3.5) * buildings_bk['buildingArea']
buildings_bk['buildingVolume'] = mm.Volume(buildings_bk, 'buildingHeight', 'buildingArea').series
buildings_bk['buildingPerimeter'] = mm.Perimeter(buildings_bk).series
buildings_bk['buildingCourtyardArea'] = mm.CourtyardArea(buildings_bk, 'buildingArea').series
buildings_bk['buildingCourtyardIndex'] = mm.CourtyardIndex(buildings_bk, 'buildingCourtyardArea', 'buildingArea').series
buildings_bk['buildingCircularCompactness'] = mm.CircularCompactness(buildings_bk, 'buildingArea').series
buildings_bk['buildingElongation'] = mm.Elongation(buildings_bk).series
buildings_bk['buildingSquareness'] = mm.Squareness(buildings_bk, verbose=False).series
buildings_bk['buildingFormFactor'] = mm.FormFactor(buildings_bk, 'buildingVolume', 'buildingArea', 'buildingHeight').series
buildings_bk['buildingVolumeFacadeRatio'] = mm.VolumeFacadeRatio(buildings_bk, 'buildingHeight', 'buildingArea', 'buildingPerimeter').series
buildings_bk['buildingCorners'] = mm.Corners(buildings_bk, verbose=False).series
buildings_bk['buildingEquivalentRectangularIndex'] = mm.EquivalentRectangularIndex(buildings_bk, 'buildingArea', 'buildingPerimeter').series
buildings_bk['buildingSharedWallsRatio'] = mm.SharedWallsRatio(buildings_bk, 'buildingPerimeter').series
buildings_bk['buildingLongestAxisLength'] = mm.LongestAxisLength(buildings_bk).series
buildings_bk['buildingShapeIndex'] = mm.ShapeIndex(buildings_bk, longest_axis='buildingLongestAxisLength', areas='buildingArea').series
buildings_bk['buildingConvexity'] = mm.Convexity(buildings_bk).series
buildings_bk['buildingFractalDimension'] = mm.FractalDimension(buildings_bk, 'buildingArea', 'buildingPerimeter').series
buildings_bk['buildingRectangularity'] = mm.Rectangularity(buildings_bk, 'buildingArea').series

centroid_corners = mm.CentroidCorners(buildings_bk, verbose=False)
buildings_bk['buildingCentroidCornersMean'] = centroid_corners.mean
buildings_bk['buildingCentroidCornersStD'] = centroid_corners.std

tessellation_bk['tessArea'] = mm.Area(tessellation_bk).series
tessellation_bk['tessCompactnessWeightedAxis'] = mm.CompactnessWeightedAxis(tessellation_bk).series
tessellation_bk['tessCircularCompactness'] = mm.CircularCompactness(tessellation_bk, 'tessArea').series
tessellation_bk['tessLongestAxisLength'] = mm.LongestAxisLength(tessellation_bk).series
tessellation_bk['tessEquivalentRectangularIndex'] = mm.EquivalentRectangularIndex(tessellation_bk, 'tessArea').series

blocks_bk['blockArea'] = mm.Area(blocks_bk).series
blocks_bk['blockPerimeter'] = mm.Perimeter(blocks_bk).series
blocks_bk['blockCircularCompactness'] = mm.CircularCompactness(blocks_bk, 'blockArea').series
blocks_bk['blockEquivalentRectangularIndex'] = mm.EquivalentRectangularIndex(blocks_bk, 'blockArea', 'blockPerimeter').series
blocks_bk['blockCompactnessWeightedAxis'] = mm.CompactnessWeightedAxis(blocks_bk, 'blockArea', 'blockPerimeter').series
blocks_bk['blockOrientation'] = mm.Orientation(blocks_bk, verbose=False).series

In [ ]:
buildings_bk['buildingOrientation'] = mm.Orientation(buildings_bk, verbose=False).series
tessellation_bk['tessOrientation'] = mm.Orientation(tessellation_bk, verbose=False).series
#buildings_bk['buildingCellAlignment'] = mm.CellAlignment(buildings_bk, tessellation_bk, 'buildingOrientation', 'tessOrientation', 'uID', 'uID').series

tessellation_bk['tessStreetAlignment'] = mm.StreetAlignment(tessellation_bk, streets_bk, 'tessOrientation', 'nID').series
buildings_bk['buildingStreetAlignment'] = mm.StreetAlignment(buildings_bk, streets_bk, 'buildingOrientation', 'nID').series

In [ ]:
profile = mm.StreetProfile(streets_bk, buildings_bk, heights='buildingHeight', distance=3)
streets_bk['streetWidth'] = profile.w
streets_bk['streetHeight'] = profile.h
streets_bk['streetProfile'] = profile.p
streets_bk['streetOpenness'] = profile.o
streets_bk['streetWidthDeviations'] = profile.wd
streets_bk['streetHeightDeviations'] = profile.hd

In [ ]:
tessellation_bk['tessCoveredAreaRatio'] = mm.AreaRatio(tessellation_bk, buildings_bk, 'tessArea', 'buildingArea', 'uID').series

In [ ]:
streets_bk['streetPerimeter'] = mm.Perimeter(streets_bk).series
streets_bk['streetLinearity'] = mm.Linearity(streets_bk).series
streets_bk["streetArea"] = mm.Reached(streets_bk, tessellation_bk, 'nID', 'nID', mode='sum', values='tessArea').series
streets_bk["streetBuildingCount"] = mm.Count(streets_bk, buildings_bk, 'nID', 'nID', weighted=True).series

In [ ]:
tessellation_bk = tessellation_bk.drop_duplicates('uID')
queen_tessellation = libpysal.weights.contiguity.Queen.from_dataframe(tessellation_bk, ids='uID', silence_warnings=True)
queen_buildings = libpysal.weights.contiguity.Queen.from_dataframe(buildings_bk, silence_warnings=True)
queen_blocks = libpysal.weights.contiguity.Queen.from_dataframe(blocks_bk, ids='bID', silence_warnings=True)
#queen_streets = libpysal.weights.contiguity.Queen.from_dataframe(streets_bk, silence_warnings=True)

queen_k3 = mm.sw_high(k=3, weights=queen_tessellation)

In [ ]:
buildings_bk['buildingAlignment'] = mm.Alignment(buildings_bk, queen_tessellation, 'uID', 'buildingOrientation', verbose=False).series
buildings_bk['buildingNeighborDistance'] = mm.NeighborDistance(buildings_bk, queen_tessellation, 'uID', verbose=False).series
buildings_bk['buildingCourtyards'] = mm.Courtyards(buildings_bk, spatial_weights=queen_buildings, verbose=False).series
buildings_bk['buildingPerimeterWall'] = mm.PerimeterWall(buildings_bk, queen_buildings, verbose=False).series

tessellation_bk['tessNeighbors'] = mm.Neighbors(tessellation_bk, queen_tessellation, 'uID', weighted=True, verbose=False).series
tessellation_bk['tessCoveredArea'] = mm.CoveredArea(tessellation_bk, queen_tessellation, 'uID', verbose=False).series

blocks_bk['blockNeighbors'] = mm.Neighbors(blocks_bk, queen_blocks, 'bID', weighted=True, verbose=False).series
blocks_bk['blockBuildingCount'] = mm.Count(blocks_bk, buildings_bk, 'bID', 'bID', weighted=True).series

In [ ]:
buildings_bk['buildingMeanInterbuildingDistance'] = mm.MeanInterbuildingDistance(buildings_bk, queen_tessellation, 'uID', queen_k3, verbose=False).series
buildings_bk['buildingBuildingAdjacency'] = mm.BuildingAdjacency(buildings_bk, queen_k3, 'uID', queen_buildings, verbose=False).series

In [ ]:
graph = mm.gdf_to_nx(streets_bk)
graph = mm.node_degree(graph)
graph = mm.closeness_centrality(graph, radius=400, distance="mm_len")
graph = mm.meshedness(graph, radius=400, distance="mm_len")
nodes_bk, streets_bk = mm.nx_to_gdf(graph)

buildings_bk["nodeID"] = mm.get_node_id(buildings_bk, nodes_bk, streets_bk, "nodeID", "nID")

In [ ]:
merged = tessellation_bk.merge(buildings_bk.drop(columns=['nID', 'bID', 'geometry']), on='uID')
merged = merged.merge(blocks_bk.drop(columns='geometry'), on='bID', how='left')
merged = merged.merge(streets_bk.drop(columns='geometry'), on='nID', how='left')
merged = merged.merge(nodes_bk.drop(columns='geometry'), on='nodeID', how='left')
merged = merged.drop(columns=['geometry'])
merged.to_csv('data/merged.csv', index=False)

In [ ]:
percentiles = []
for column in merged.columns.drop(["uID", "nodeID", "nID", 'eID', 'bID', 'mm_len', 'node_start', 'node_end']):
    perc = mm.Percentiles(merged, column, queen_k3, "uID", verbose=False).frame
    perc.columns = [f"{column}_" + str(x) for x in perc.columns]
    percentiles.append(perc)
    
percentiles = pd.concat(percentiles, axis=1)
percentiles.to_csv('data/percentiles.csv', index=False)